# NeurIPS Polymers — Integrated Pipeline (Phase1 ⨯ Phase3 + Selection + Scaling)

- Input: `competition/train.csv`, `competition/test.csv`, `competition/sample_submission.csv`
- Features: Mordred(2D) + RDKit basics + Physics pack + Morgan r2 counts + Morgan r3 presence + MACCS (cached)
- Cleaning: convert Mordred error objects → NaN; numeric-only parquet; median impute; **variance & correlation filters**
- CV: **GroupKFold(groups=canonical_smiles)**; per-target masking
- Pruning: **gain‑prune** (union top‑K per target) after the prefilters
- Models: LightGBM + CatBoost + Kernel SVR (Tanimoto) ; **3 seeds**
- Blending: **OOF NNLS** (non-negative), per target
- Post: range clipping per target; leaderboard‑aligned **mMAE** (unweighted masked)
- Additions: **Feature Selection (variance+correlation)** and **Feature Scaling (StandardScaler on pruned features)**
- Optional: MIL conformer pooling for Tg/Rg (toggle later)


In [14]:
# Expected runtime: < 1 min
# Purpose: imports, paths, seeds, basic env logging
# Outputs: Printed paths; output dirs created
# Sanity: assert competition files exist

import os, sys, json, time, random, warnings, glob, hashlib, gc
from pathlib import Path
import numpy as np
import pandas as pd
warnings.filterwarnings("ignore")

# Repro
SEEDS = [111, 222, 333]
def seed_everything(seed=222):
    random.seed(seed); np.random.seed(seed)
    try:
        import torch
        torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
    except Exception:
        pass

# Paths (match your tree)
ROOT = Path(".")
DATA_DIR = ROOT / "competition"
CACHE_DIR = ROOT / "artifacts" / "models"
OUT_DIR = ROOT / "outputs"
OUT_PREDS  = OUT_DIR / "predictions"
for d in [CACHE_DIR, OUT_DIR, OUT_PREDS]:
    d.mkdir(parents=True, exist_ok=True)

TRAIN_CSV = DATA_DIR / "train.csv"
TEST_CSV  = DATA_DIR / "test.csv"
SAMPLE_CSV= DATA_DIR / "sample_submission.csv"

print("DATA_DIR:", DATA_DIR.resolve())
print("CACHE_DIR:", CACHE_DIR.resolve())
print("OUT_PREDS:", OUT_PREDS.resolve())
print("SEEDS:", SEEDS)

# Sanity
assert TRAIN_CSV.exists() and TEST_CSV.exists() and SAMPLE_CSV.exists(), "Missing files in ./competition/"
print("✓ Found competition files")


DATA_DIR: /home/rohit/Desktop/NeurIPS/competition
CACHE_DIR: /home/rohit/Desktop/NeurIPS/artifacts/models
OUT_PREDS: /home/rohit/Desktop/NeurIPS/outputs/predictions
SEEDS: [111, 222, 333]
✓ Found competition files


In [15]:
# Expected runtime: < 10 s
# Purpose: declare targets, masked per-target MAE + unweighted mMAE
# Outputs: compute_mmae, masked_mae_per_target
# Sanity: toy example mMAE within [0,1)

from sklearn.metrics import mean_absolute_error

TARGETS = ["Tg", "FFV", "Tc", "Density", "Rg"]

def masked_mae_per_target(y_true_df: pd.DataFrame, y_pred_df: pd.DataFrame):
    maes = {}
    for t in TARGETS:
        if t not in y_true_df.columns: continue
        yt = y_true_df[t].to_numpy()
        yp = y_pred_df[t].to_numpy()
        mask = (~np.isnan(yt)) & np.isfinite(yp)
        maes[t] = mean_absolute_error(yt[mask], yp[mask]) if mask.sum() else np.nan
    return maes

def compute_mmae(y_true_df: pd.DataFrame, y_pred_df: pd.DataFrame):
    maes = masked_mae_per_target(y_true_df, y_pred_df)
    vals = [v for v in maes.values() if v==v]
    return float(np.mean(vals)) if vals else np.nan, maes

# Sanity
toy_true = pd.DataFrame({"Tg":[1,2,np.nan], "FFV":[0.1,0.2,0.3]})
toy_pred = pd.DataFrame({"Tg":[1.2,1.8,5.0], "FFV":[0.0,0.2,0.4]})
m, d = compute_mmae(toy_true, toy_pred)
print("Toy mMAE:", round(m,4), "per-target:", {k:round(v,4) for k,v in d.items() if v==v})
assert 0 <= m < 1.0
print("✓ Metric OK")


Toy mMAE: 0.1333 per-target: {'Tg': 0.2, 'FFV': 0.0667}
✓ Metric OK


In [16]:
# --- Cell 3: Feature Engineering (Revised with physics-informed features) ---
# Expected runtime: <2 minutes (depends on dataset size)

from rdkit import Chem
from rdkit.Chem import Descriptors, rdMolDescriptors

def featurize_molecule(smiles: str) -> dict:
    """
    Extracts RDKit descriptors for a single molecule, including
    both generic and physics-informed features.
    
    Returns:
        dict of feature_name: value
    """
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return {}

    features = {
        "MolWt": Descriptors.MolWt(mol),
        "LogP": Descriptors.MolLogP(mol),
        "NumRotatableBonds": Descriptors.NumRotatableBonds(mol),   # flexibility
        "FractionCSP3": rdMolDescriptors.CalcFractionCSP3(mol),    # 3D saturation
        "AromaticProportion": sum(1 for atom in mol.GetAtoms() if atom.GetIsAromatic()) / mol.GetNumAtoms(),
        "NumHDonors": rdMolDescriptors.CalcNumHBD(mol),            # H-bond donors
        "NumHAcceptors": rdMolDescriptors.CalcNumHBA(mol),         # H-bond acceptors
        "TPSA": rdMolDescriptors.CalcTPSA(mol),                    # Polar surface area
    }
    return features

# Apply to full dataset
def featurize_dataframe(df, smiles_col="SMILES"):
    feature_rows = []
    for smi in tqdm(df[smiles_col], desc="Featurizing molecules"):
        feature_rows.append(featurize_molecule(smi))
    return pd.DataFrame(feature_rows)

train_feats = featurize_dataframe(train_df, "SMILES")
test_feats  = featurize_dataframe(test_df, "SMILES")

print(f"Feature matrix shape: train={train_feats.shape}, test={test_feats.shape}")


Featurizing molecules:   0%|          | 0/7973 [00:00<?, ?it/s]

Featurizing molecules: 100%|██████████| 3/3 [00:00<00:00, 1323.26it/s]

Feature matrix shape: train=(7973, 8), test=(3, 8)


In [17]:
# Expected runtime: define-only (< 10 s)
# Purpose: future-proof cleaners + builders (RDKit basics, physics, fingerprints, Mordred)
# Outputs: functions
# Sanity: none (defs only)

def clean_object_columns(df: pd.DataFrame) -> pd.DataFrame:
    """Ensure numeric-only columns: non-numeric → NaN, inf → NaN, coerce to float."""
    def _num_or_nan(x):
        if x is None: return np.nan
        if isinstance(x, (int, float, np.integer, np.floating)): return x
        # Mordred Error or any other non-numeric object → NaN
        return np.nan
    df = df.applymap(_num_or_nan)
    df = df.replace([np.inf, -np.inf], np.nan)
    df = df.apply(pd.to_numeric, errors="coerce")
    return df

from rdkit.Chem import Descriptors, rdMolDescriptors as rdm, AllChem, MACCSkeys
from mordred import Calculator, descriptors

def rdkit_basic(mol):
    if mol is None: return np.array([np.nan]*8, float)
    return np.array([
        Descriptors.MolWt(mol),
        Descriptors.TPSA(mol),
        Descriptors.MolLogP(mol),
        rdm.CalcNumRings(mol),
        rdm.CalcNumAromaticRings(mol),
        rdm.CalcNumAliphaticRings(mol),
        rdm.CalcNumHBD(mol),
        rdm.CalcNumHBA(mol),
    ], float)

def gasteiger_stats(mol):
    try:
        Chem.rdPartialCharges.ComputeGasteigerCharges(mol)
        vals=[]
        for a in mol.GetAtoms():
            q = a.GetDoubleProp("_GasteigerCharge") if a.HasProp("_GasteigerCharge") else np.nan
            vals.append(q if q==q else np.nan)
        arr = np.array(vals, float)
        return np.array([np.nanmean(arr), np.nanstd(arr), np.nanmax(arr), np.nanmin(arr)], float)
    except Exception:
        return np.array([np.nan, np.nan, np.nan, np.nan], float)

def physics_pack(mol):
    if mol is None: return np.array([np.nan]*14, float)
    rot = rdm.CalcNumRotatableBonds(mol)
    frac_sp3 = rdm.CalcFractionCSP3(mol)
    rings = rdm.CalcNumRings(mol)
    arom = rdm.CalcNumAromaticRings(mol)
    aliph = rdm.CalcNumAliphaticRings(mol)
    heavy = Descriptors.HeavyAtomCount(mol)
    exactmw = Descriptors.ExactMolWt(mol)
    bertz = Descriptors.BertzCT(mol)
    mr = Descriptors.MolMR(mol)
    arom_prop = (arom / rings) if rings > 0 else 0.0
    charges = gasteiger_stats(mol)
    base = np.array([rot, frac_sp3, rings, arom, aliph, heavy, exactmw, bertz, mr, arom_prop], float)
    return np.hstack([base, charges])  # ~14

def morgan_r2_counts(mol, nBits=1024):
    if mol is None: return np.zeros(nBits, float)
    fp = AllChem.GetHashedMorganFingerprint(mol, radius=2, nBits=nBits)
    arr = np.zeros((nBits,), int)
    for idx, cnt in fp.GetNonzeroElements().items():
        arr[idx % nBits] = cnt
    return arr.astype(float)

def morgan_r3_presence(mol, nBits=1024):
    if mol is None: return np.zeros(nBits, float)
    bv = AllChem.GetMorganFingerprintAsBitVect(mol, radius=3, nBits=nBits)
    return np.array([int(bv.GetBit(i)) for i in range(nBits)], float)

def maccs_bits(mol):
    if mol is None: return np.zeros(166, float)
    bv = MACCSkeys.GenMACCSKeys(mol)
    return np.array([int(bv.GetBit(i)) for i in range(bv.GetNumBits())], float)

# Mordred calculator
MORDRED_2D = Calculator(descriptors, ignore_3D=True)


In [18]:
# Cell 5: Mordred descriptors
# Expected runtime: 10–40 min (first time); ~10–30 s if cached
# Purpose: prefer cached; else compute; always clean & align before saving
# Outputs: desc_comptrain_fastv1.parquet, desc_comptest_fastv1.parquet aligned
# Sanity: no object dtypes; aligned columns

def compute_mordred_df(smiles_list):
    mols = [Chem.MolFromSmiles(s) for s in smiles_list]
    df = MORDRED_2D.pandas(mols)
    df = clean_object_columns(df)
    return df

tr_path = CACHE_DIR / "desc_comptrain_fastv1.parquet"
te_path = CACHE_DIR / "desc_comptest_fastv1.parquet"

if tr_path.exists() and te_path.exists():
    mordred_train = pd.read_parquet(tr_path)
    mordred_test  = pd.read_parquet(te_path)
    print("Loaded cached Mordred parquet.")
else:
    print("Computing Mordred (train/test)…")
    mordred_train = compute_mordred_df(train_df["SMILES_CANON"].tolist())
    mordred_test  = compute_mordred_df(test_df["SMILES_CANON"].tolist())

    # Drop constants based on train only
    nunq = mordred_train.nunique(dropna=True)
    keep_cols = nunq[nunq > 1].index
    mordred_train = mordred_train[keep_cols]

    # Save raw versions (will align below)
    mordred_train.to_parquet(tr_path)
    mordred_test.to_parquet(te_path)
    print("Cached Mordred to parquet.")

# 🔑 Always align after load or compute
all_cols = sorted(set(mordred_train.columns) | set(mordred_test.columns))
mordred_train = mordred_train.reindex(columns=all_cols)
mordred_test  = mordred_test.reindex(columns=all_cols)

# Clean again to be safe
mordred_train = clean_object_columns(mordred_train)
mordred_test  = clean_object_columns(mordred_test)

# Sanity checks
train_cols = set(mordred_train.columns)
test_cols  = set(mordred_test.columns)

print("Train-only columns:", train_cols - test_cols)
print("Test-only columns:", test_cols - train_cols)
print("Common columns:", len(train_cols & test_cols))

assert mordred_train.shape[1] == mordred_test.shape[1], "Mordred train/test not aligned!"
assert mordred_train.select_dtypes(include="object").empty
assert mordred_test.select_dtypes(include="object").empty
print("✓ Mordred ready:", mordred_train.shape, mordred_test.shape)


Loaded cached Mordred parquet.
Train-only columns: set()
Test-only columns: set()
Common columns: 782
✓ Mordred ready: (7973, 782) (3, 782)


In [19]:
# Expected runtime: 5–20 min total (parallelized)
# Purpose: compute missing blocks & cache (numeric-only)
# Outputs: rdkit8_*.parquet, phys_*.parquet, morgan_r3_*.parquet, maccs_*.parquet, morgan_r2_*.parquet
# Sanity: dims match train/test; numeric-only

from joblib import Parallel, delayed

mols_tr = [Chem.MolFromSmiles(s) for s in train_df["SMILES_CANON"]]
mols_te = [Chem.MolFromSmiles(s) for s in test_df["SMILES_CANON"]]

def block_from(func, mols, name, ncols):
    arr = Parallel(n_jobs=os.cpu_count()//2 or 4)(delayed(func)(m) for m in mols)
    df = pd.DataFrame(np.vstack(arr), columns=[f"{name}_{i}" for i in range(ncols)])
    return clean_object_columns(df)

blocks = {
    "rdkit8": (rdkit_basic, 8),
    "phys":   (physics_pack, 14),
    "morgan_r3p": (lambda m: morgan_r3_presence(m, 1024), 1024),
    "maccs": (maccs_bits, 166),
    "morgan_r2c": (lambda m: morgan_r2_counts(m, 1024), 1024),
}

for key, (fn, n) in blocks.items():
    tr_p = CACHE_DIR / f"{key}_train.parquet"
    te_p = CACHE_DIR / f"{key}_test.parquet"
    if tr_p.exists() and te_p.exists():
        print(f"Loaded cached {key}.")
        continue
    print(f"Computing {key} …")
    df_tr = block_from(fn, mols_tr, key, n)
    df_te = block_from(fn, mols_te, key, n)
    df_tr.to_parquet(tr_p); df_te.to_parquet(te_p)

# Load all blocks (fresh or cached)
rdkit8_tr = pd.read_parquet(CACHE_DIR/"rdkit8_train.parquet")
rdkit8_te = pd.read_parquet(CACHE_DIR/"rdkit8_test.parquet")
phys_tr   = pd.read_parquet(CACHE_DIR/"phys_train.parquet")
phys_te   = pd.read_parquet(CACHE_DIR/"phys_test.parquet")
m3_tr     = pd.read_parquet(CACHE_DIR/"morgan_r3p_train.parquet")
m3_te     = pd.read_parquet(CACHE_DIR/"morgan_r3p_test.parquet")
m2_tr     = pd.read_parquet(CACHE_DIR/"morgan_r2c_train.parquet")
m2_te     = pd.read_parquet(CACHE_DIR/"morgan_r2c_test.parquet")
maccs_tr  = pd.read_parquet(CACHE_DIR/"maccs_train.parquet")
maccs_te  = pd.read_parquet(CACHE_DIR/"maccs_test.parquet")

# Sanity
assert rdkit8_tr.shape[1]==8 and rdkit8_te.shape[1]==8
assert phys_tr.shape[1]==phys_te.shape[1]==14
assert m3_tr.shape[1]==m3_te.shape[1]==1024
assert m2_tr.shape[1]==m2_te.shape[1]==1024
assert maccs_tr.shape[1]==maccs_te.shape[1]==166
print("✓ RDKit/Physics/Morgan/MACCS blocks ready")


Computing rdkit8 …
Computing phys …


/tmp/ipykernel_658241/291762141.py:42: RuntimeWarning: Mean of empty slice
/home/rohit/.local/lib/python3.10/site-packages/numpy/lib/nanfunctions.py:1879: RuntimeWarning: Degrees of freedom <= 0 for slice.
  var = nanvar(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
/tmp/ipykernel_658241/291762141.py:42: RuntimeWarning: All-NaN slice encountered
/home/rohit/.local/lib/python3.10/site-packages/numpy/lib/nanfunctions.py:1741: RuntimeWarning: invalid value encountered in subtract
  np.subtract(arr, avg, out=arr, casting='unsafe', where=where)
/home/rohit/.local/lib/python3.10/site-packages/numpy/lib/nanfunctions.py:1741: RuntimeWarning: invalid value encountered in subtract
  np.subtract(arr, avg, out=arr, casting='unsafe', where=where)
/tmp/ipykernel_658241/291762141.py:42: RuntimeWarning: Mean of empty slice
/home/rohit/.local/lib/python3.10/site-packages/numpy/lib/nanfunctions.py:1879: RuntimeWarning: Degrees of freedom <= 0 for slice.
  var = nanvar(a, axis=axis, dtype=dtype, out=out

Computing morgan_r3p …


[15:12:20] DEPRECATION WARNING: please use MorganGenerator
[15:12:20] DEPRECATION WARNING: please use MorganGenerator
[15:12:20] DEPRECATION WARNING: please use MorganGenerator
[15:12:20] DEPRECATION WARNING: please use MorganGenerator
[15:12:20] DEPRECATION WARNING: please use MorganGenerator
[15:12:20] DEPRECATION WARNING: please use MorganGenerator
[15:12:20] DEPRECATION WARNING: please use MorganGenerator
[15:12:20] DEPRECATION WARNING: please use MorganGenerator
[15:12:20] DEPRECATION WARNING: please use MorganGenerator
[15:12:20] DEPRECATION WARNING: please use MorganGenerator
[15:12:20] DEPRECATION WARNING: please use MorganGenerator
[15:12:20] DEPRECATION WARNING: please use MorganGenerator
[15:12:20] DEPRECATION WARNING: please use MorganGenerator
[15:12:20] DEPRECATION WARNING: please use MorganGenerator
[15:12:20] DEPRECATION WARNING: please use MorganGenerator
[15:12:20] DEPRECATION WARNING: please use MorganGenerator
[15:12:20] DEPRECATION WARNING: please use MorganGenerat

Loaded cached maccs.
Computing morgan_r2c …


[15:12:23] DEPRECATION WARNING: please use MorganGenerator
[15:12:23] DEPRECATION WARNING: please use MorganGenerator
[15:12:23] DEPRECATION WARNING: please use MorganGenerator
[15:12:23] DEPRECATION WARNING: please use MorganGenerator
[15:12:23] DEPRECATION WARNING: please use MorganGenerator
[15:12:23] DEPRECATION WARNING: please use MorganGenerator
[15:12:23] DEPRECATION WARNING: please use MorganGenerator
[15:12:23] DEPRECATION WARNING: please use MorganGenerator
[15:12:23] DEPRECATION WARNING: please use MorganGenerator
[15:12:23] DEPRECATION WARNING: please use MorganGenerator
[15:12:23] DEPRECATION WARNING: please use MorganGenerator
[15:12:23] DEPRECATION WARNING: please use MorganGenerator
[15:12:23] DEPRECATION WARNING: please use MorganGenerator
[15:12:23] DEPRECATION WARNING: please use MorganGenerator
[15:12:23] DEPRECATION WARNING: please use MorganGenerator
[15:12:23] DEPRECATION WARNING: please use MorganGenerator
[15:12:23] DEPRECATION WARNING: please use MorganGenerat

AssertionError: 

In [20]:
# Expected runtime: 20–60 s
# Purpose: assemble feature matrix; **add feature selection prefilters**; median impute; align columns
# Outputs: Xtr_pref, Xte_pref, feature_names_pref.json, medians.npy
# Sanity: no NaNs; dims match; filters applied

# Combine
Xtr_full = pd.concat([mordred_train, rdkit8_tr, phys_tr, m2_tr, m3_tr, maccs_tr], axis=1)
Xte_full = pd.concat([mordred_test,  rdkit8_te, phys_te, m2_te, m3_te, maccs_te], axis=1)

# 7a) Variance threshold (drop near-constant features)
var = Xtr_full.var(axis=0, skipna=True)
keep_var = var > 1e-12  # extremely conservative threshold
Xtr_v = Xtr_full.loc[:, keep_var]
Xte_v = Xte_full.loc[:, keep_var]

# 7b) High-correlation filter (drop one of pairs with |corr| > 0.995)
# Use a small random subset for speed; fallback to full if small data
sub_idx = np.random.RandomState(0).choice(len(Xtr_v), size=min(500, len(Xtr_v)), replace=False)
corr = np.abs(pd.DataFrame(Xtr_v.values[sub_idx]).corr())
upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
drop_cols_idx = [col for col in range(upper.shape[1]) if any(upper.iloc[:, col] > 0.995)]
keep_corr_mask = np.ones(Xtr_v.shape[1], dtype=bool)
keep_corr_mask[drop_cols_idx] = False

Xtr_pref = Xtr_v.iloc[:, keep_corr_mask]
Xte_pref = Xte_v.iloc[:, keep_corr_mask]

# Median impute (train medians)
train_medians = Xtr_pref.median(axis=0).astype(float)
Xtr_pref = Xtr_pref.fillna(train_medians)
Xte_pref = Xte_pref.fillna(train_medians)

feature_names_pref = list(Xtr_pref.columns)
np.save(OUT_PREDS / "train_medians.npy", train_medians.to_numpy())
json.dump(feature_names_pref, open(OUT_PREDS / "feature_names_pref.json","w"))

print("Prefiltered shapes:", Xtr_pref.shape, Xte_pref.shape)

# Sanity
assert Xtr_pref.shape[1] == Xte_pref.shape[1] == len(feature_names_pref)
assert not Xtr_pref.isna().any().any() and not Xte_pref.isna().any().any()
print("✓ Prefilter (variance+correlation) + impute + align complete")


Prefiltered shapes: (7973, 2641) (3, 2641)
✓ Prefilter (variance+correlation) + impute + align complete


In [21]:
# Expected runtime: 5–12 min
# Purpose: union top-K features per target using fast LightGBM; apply mask
# Outputs: keep_mask.npy, pruned_feature_names.json, X_train_pruned, X_test_pruned
# Sanity: pruned dims < prefiltered; no NaNs

import lightgbm as lgb
from sklearn.model_selection import KFold

K_PER_TARGET = 600
FOLDS_FOR_PRUNE = 3
feat_votes = np.zeros(len(feature_names_pref), float)

y_all = train_df[TARGETS].copy()
kf = KFold(n_splits=FOLDS_FOR_PRUNE, shuffle=True, random_state=22222)

for t in TARGETS:
    y = y_all[t].to_numpy()
    mask = ~np.isnan(y)
    if mask.sum() < 50:
        continue
    gains = np.zeros(len(feature_names_pref), float)
    for tr_idx, va_idx in kf.split(Xtr_pref.values[mask], y[mask]):
        dtr = lgb.Dataset(Xtr_pref.values[mask][tr_idx], label=y[mask][tr_idx])
        dva = lgb.Dataset(Xtr_pref.values[mask][va_idx], label=y[mask][va_idx])
        params = dict(objective="regression", metric="mae", learning_rate=0.08, num_leaves=256,
                      feature_fraction=0.8, bagging_fraction=0.8, bagging_freq=1, verbose=-1, seed=22222)
        m = lgb.train(params, dtr, num_boost_round=150, valid_sets=[dva],
                      callbacks=[lgb.early_stopping(30, verbose=False)])
        gains += m.feature_importance(importance_type="gain")
    topk = np.argsort(-gains)[:K_PER_TARGET]
    feat_votes[topk] += 1.0

keep_mask = feat_votes > 0
pruned_names = [n for n, k in zip(feature_names_pref, keep_mask) if k]
np.save(OUT_PREDS / "keep_mask.npy", keep_mask.astype(np.uint8))
json.dump(pruned_names, open(OUT_PREDS / "pruned_feature_names.json","w"))

X_train_pruned = Xtr_pref[pruned_names].copy()
X_test_pruned  = Xte_pref[pruned_names].copy()

print("Kept:", X_train_pruned.shape[1], "of", len(feature_names_pref))

# Sanity
assert X_train_pruned.shape[1] < len(feature_names_pref)
assert X_train_pruned.shape[1] == X_test_pruned.shape[1]
assert not X_train_pruned.isna().any().any() and not X_test_pruned.isna().any().any()
print("✓ Gain‑prune OK")


Kept: 938 of 2641
✓ Gain‑prune OK


In [22]:
# Expected runtime: < 5 s
# Purpose: **new addition** — fit StandardScaler on pruned features; produce scaled copies
# Outputs: X_train_pruned_scaled, X_test_pruned_scaled, scaler.pkl
# Sanity: shapes match; no NaNs

from sklearn.preprocessing import StandardScaler
import pickle as pkl

scaler = StandardScaler(with_mean=True, with_std=True)
X_train_pruned_scaled = pd.DataFrame(
    scaler.fit_transform(X_train_pruned.values), index=X_train_pruned.index, columns=X_train_pruned.columns
)
X_test_pruned_scaled = pd.DataFrame(
    scaler.transform(X_test_pruned.values), index=X_test_pruned.index, columns=X_test_pruned.columns
)

with open(OUT_PREDS/"scaler.pkl","wb") as f:
    pkl.dump(scaler, f)

print("Scaled shapes:", X_train_pruned_scaled.shape, X_test_pruned_scaled.shape)

# Sanity
assert X_train_pruned_scaled.shape == X_train_pruned.shape
assert not np.isnan(X_train_pruned_scaled.values).any() and not np.isnan(X_test_pruned_scaled.values).any()
print("✓ Scaling ready (trees will use unscaled; kernels use scaled/binary as needed)")


Scaled shapes: (7973, 938) (3, 938)
✓ Scaling ready (trees will use unscaled; kernels use scaled/binary as needed)


In [23]:
# Expected runtime: < 5 s
# Purpose: build leakage-safe folds by canonical SMILES
# Outputs: FOLDS list persisted
# Sanity: no group overlap per fold

from sklearn.model_selection import GroupKFold
N_SPLITS = 5
groups = train_df["SMILES_CANON"].to_numpy()
gkf = GroupKFold(n_splits=N_SPLITS)
FOLDS = [(tr, va) for tr, va in gkf.split(X_train_pruned, y=train_df.index.values, groups=groups)]
json.dump([(len(a), len(b)) for a,b in FOLDS], open(OUT_PREDS/"fold_sizes.json","w"))

def no_overlap(a,b):
    return len(set(groups[a]) & set(groups[b])) == 0
assert all(no_overlap(tr,va) for tr,va in FOLDS), "Group leakage!"
print("✓ GroupKFold ready; fold sizes:", [(len(a), len(b)) for a,b in FOLDS])


✓ GroupKFold ready; fold sizes: [(6378, 1595), (6378, 1595), (6378, 1595), (6379, 1594), (6379, 1594)]


In [24]:
# Expected runtime: 25–60 min
# Purpose: per-target training (trees use UNscaled pruned features); collect OOF/TEST; seed-average
# Outputs: oof_lgb.csv, test_lgb.csv, oof_cat.csv, test_cat.csv
# Sanity: finite predictions; printed per-target OOF MAE; mMAE

import lightgbm as lgb
from catboost import CatBoostRegressor

def fit_lgb_cat(X_tr, y_df, X_te, folds, seeds):
    oof_lgb = {t: np.full(len(y_df), np.nan) for t in TARGETS}
    oof_cat = {t: np.full(len(y_df), np.nan) for t in TARGETS}
    te_lgb  = {t: np.zeros(X_te.shape[0], float) for t in TARGETS}
    te_cat  = {t: np.zeros(X_te.shape[0], float) for t in TARGETS}

    for seed in seeds:
        seed_everything(seed)
        for t in TARGETS:
            y = y_df[t].to_numpy()
            mask = ~np.isnan(y)
            if mask.sum() < 30:
                continue
            oof_lgb_t = np.full(len(y), np.nan); oof_cat_t = np.full(len(y), np.nan)
            te_lgb_t  = np.zeros(X_te.shape[0], float); te_cat_t  = np.zeros(X_te.shape[0], float)

            for fold_id, (tr_idx, va_idx) in enumerate(folds):
                tr = tr_idx[mask[tr_idx]]; va = va_idx[mask[va_idx]]
                if len(tr)==0 or len(va)==0: continue

                # LightGBM
                dtr = lgb.Dataset(X_tr.values[tr], label=y[tr])
                dva = lgb.Dataset(X_tr.values[va], label=y[va])
                params = dict(objective="regression", metric="mae",
                              learning_rate=0.05, num_leaves=128,
                              feature_fraction=0.8, bagging_fraction=0.8, bagging_freq=5,
                              verbose=-1, seed=seed)
                m_lgb = lgb.train(params, dtr, num_boost_round=6000,
                                  valid_sets=[dva], valid_names=["val"],
                                  callbacks=[lgb.early_stopping(200, verbose=False)])
                oof_lgb_t[va] = m_lgb.predict(X_tr.values[va], num_iteration=m_lgb.best_iteration)
                te_lgb_t += m_lgb.predict(X_te.values, num_iteration=m_lgb.best_iteration) / len(folds)

                # CatBoost
                m_cat = CatBoostRegressor(
                    loss_function="MAE", depth=8, l2_leaf_reg=3.0, learning_rate=0.05,
                    iterations=8000, od_type="Iter", od_wait=300, random_seed=seed, verbose=False
                )
                m_cat.fit(X_tr.values[tr], y[tr], eval_set=(X_tr.values[va], y[va]))
                oof_cat_t[va] = m_cat.predict(X_tr.values[va])
                te_cat_t += m_cat.predict(X_te.values) / len(folds)

            # accumulate across seeds
            oof_lgb[t] = np.nanmean(np.vstack([oof_lgb[t], oof_lgb_t]), axis=0)
            oof_cat[t] = np.nanmean(np.vstack([oof_cat[t], oof_cat_t]), axis=0)
            te_lgb[t]  += te_lgb_t / len(seeds)
            te_cat[t]  += te_cat_t / len(seeds)

            print(f"[Seed {seed}] {t} — LGB OOF MAE:",
                  mean_absolute_error(y[~np.isnan(y)], oof_lgb_t[~np.isnan(y)]),
                  "| CAT OOF MAE:",
                  mean_absolute_error(y[~np.isnan(y)], oof_cat_t[~np.isnan(y)]))

    pd.DataFrame(oof_lgb).to_csv(OUT_PREDS/"oof_lgb.csv", index=False)
    pd.DataFrame(oof_cat).to_csv(OUT_PREDS/"oof_cat.csv", index=False)
    pd.DataFrame(te_lgb).to_csv(OUT_PREDS/"test_lgb.csv", index=False)
    pd.DataFrame(te_cat).to_csv(OUT_PREDS/"test_cat.csv", index=False)
    return oof_lgb, oof_cat, te_lgb, te_cat

oof_lgb, oof_cat, test_lgb, test_cat = fit_lgb_cat(X_train_pruned, train_df[TARGETS], X_test_pruned, FOLDS, SEEDS)

mmae_lgb, _ = compute_mmae(train_df[TARGETS], pd.DataFrame(oof_lgb))
mmae_cat, _ = compute_mmae(train_df[TARGETS], pd.DataFrame(oof_cat))
print("OOF mMAE — LGB:", round(mmae_lgb,5), " | CAT:", round(mmae_cat,5))

# Sanity
assert np.isfinite(pd.DataFrame(test_lgb).to_numpy()).all()
assert np.isfinite(pd.DataFrame(test_cat).to_numpy()).all()
print("✓ Trees trained and saved")


[Seed 111] Tg — LGB OOF MAE: 48.66829444332561 | CAT OOF MAE: 51.3993458362713
[Seed 111] FFV — LGB OOF MAE: 0.005885418886004727 | CAT OOF MAE: 0.005885537279556739
[Seed 111] Tc — LGB OOF MAE: 0.025289819800913788 | CAT OOF MAE: 0.02586353115533605
[Seed 111] Density — LGB OOF MAE: 0.025511844600077646 | CAT OOF MAE: 0.028626446224413835
[Seed 111] Rg — LGB OOF MAE: 1.5315099858810282 | CAT OOF MAE: 1.6005589856505504
[Seed 222] Tg — LGB OOF MAE: 50.00138892452361 | CAT OOF MAE: 51.59090343957066
[Seed 222] FFV — LGB OOF MAE: 0.005937306817490746 | CAT OOF MAE: 0.005889407236848588
[Seed 222] Tc — LGB OOF MAE: 0.02561339743020954 | CAT OOF MAE: 0.02512861376215604
[Seed 222] Density — LGB OOF MAE: 0.024740790617670008 | CAT OOF MAE: 0.027771499366418716
[Seed 222] Rg — LGB OOF MAE: 1.5670147146784832 | CAT OOF MAE: 1.6056387062608253
[Seed 333] Tg — LGB OOF MAE: 48.94894924735948 | CAT OOF MAE: 51.855789222727346
[Seed 333] FFV — LGB OOF MAE: 0.005937229214059329 | CAT OOF MAE: 0.005

In [25]:
# Expected runtime: 10–20 min
# Purpose: extra diverse model family using binary features (MACCS + Morgan r3 presence)
# Outputs: oof_kernel.csv, test_kernel.csv
# Sanity: finite predictions; mMAE printed

from sklearn.svm import SVR

# Binary-only matrices from cached blocks (aligned to pruned columns if subset)
# Here we re-extract the binary subset that survived prefilter (if any), else use full binary blocks
bin_cols_in_pruned = [c for c in X_train_pruned.columns if c.startswith("maccs_") or c.startswith("morgan_r3p_")]
if bin_cols_in_pruned:
    Xb_tr = X_train_pruned[bin_cols_in_pruned].values
    Xb_te = X_test_pruned[bin_cols_in_pruned].values
else:
    Xb_tr = np.hstack([m3_tr.values, maccs_tr.values])
    Xb_te = np.hstack([m3_te.values, maccs_te.values])

# Ensure binary
Xb_tr = (Xb_tr > 0.5).astype(np.float32)
Xb_te = (Xb_te > 0.5).astype(np.float32)

def tanimoto_kernel(A, B):
    inter = A @ B.T
    sumA = A.sum(axis=1)[:,None]; sumB = B.sum(axis=1)[None,:]
    denom = (sumA + sumB - inter) + 1e-8
    return inter / denom

oof_k = {t: np.full(len(train_df), np.nan) for t in TARGETS}
tst_k = {t: np.zeros(len(test_df), dtype=float) for t in TARGETS}

for t in TARGETS:
    y = train_df[t].to_numpy()
    mask = ~np.isnan(y)
    if mask.sum() < 30:
        continue
    oof_t = np.full(len(y), np.nan)
    pred_te = np.zeros(len(test_df), dtype=float)
    for tr_idx, va_idx in FOLDS:
        tr = tr_idx[mask[tr_idx]]; va = va_idx[mask[va_idx]]
        if len(tr)==0 or len(va)==0: continue
        K_tr = tanimoto_kernel(Xb_tr[tr], Xb_tr[tr])
        K_va = tanimoto_kernel(Xb_tr[va], Xb_tr[tr])
        K_te = tanimoto_kernel(Xb_te, Xb_tr[tr])
        svr = SVR(C=3.0, epsilon=0.05, kernel='precomputed')
        svr.fit(K_tr, y[tr])
        oof_t[va] = svr.predict(K_va)
        pred_te += svr.predict(K_te) / len(FOLDS)
    oof_k[t] = oof_t
    tst_k[t] = pred_te
    print(f"[Kernel] {t} OOF MAE = {mean_absolute_error(y[~np.isnan(y)], oof_t[~np.isnan(y)]):.4f}")

pd.DataFrame(oof_k).to_csv(OUT_PREDS / "oof_kernel.csv", index=False)
pd.DataFrame(tst_k).to_csv(OUT_PREDS / "test_kernel.csv", index=False)

mmae_k, _ = compute_mmae(train_df[TARGETS], pd.DataFrame(oof_k))
print("Kernel OOF mMAE:", round(mmae_k,5))

# Sanity
assert np.isfinite(pd.DataFrame(tst_k).to_numpy()).all()
print("✓ Kernel baseline done")


AttributeError: 'int' object has no attribute 'startswith'